I. Trích embeddings từ text

In [1]:
!pip install transformers scikit-learn pandas numpy torch --quiet

In [2]:
import os
import pandas as pd
import re
from tqdm import tqdm
from transformers import DistilBertTokenizer, DistilBertModel
import torch
import numpy as np

In [3]:
# Đường dẫn
dataset_dir = r"C:\Users\Acer\0. CAPSTONE\dataset"
embedding_dir = os.path.join(dataset_dir, "embeddings", "text_embeddings")
os.makedirs(embedding_dir, exist_ok=True)

# Load model
tokenizer = DistilBertTokenizer.from_pretrained('distilbert-base-uncased')
model = DistilBertModel.from_pretrained('distilbert-base-uncased')
model.eval()

# Tạo dictionary lưu tất cả embedding
all_text_embeddings = {}

# Biến flag để kiểm tra 1 session đầu tiên
first_session_checked = False

In [4]:
file_test = r"C:\Users\Acer\0. CAPSTONE\dataset\301_P\301_TRANSCRIPT.csv"

# Thử đọc bằng auto-separator (mặc định là dấu phẩy)
df = pd.read_csv(file_test)

print(df.head())  # In 5 dòng đầu
print(df.columns)  # Xem tên các cột

   start_time  stop_time      speaker  \
0      29.428     35.888        Ellie   
1      32.738     33.068  Participant   
2      36.598     40.948        Ellie   
3      42.088     42.518  Participant   
4      42.358     51.738        Ellie   

                                               value  
0  hi i'm ellie thanks for coming in today i was ...  
1                                          thank you  
2  think of me as a friend i don't judge i can't ...  
3                                              mmm k  
4  i'm here to learn about people and would love ...  
Index(['start_time', 'stop_time', 'speaker', 'value'], dtype='object')


In [5]:
def load_all_transcripts(dataset_dir):
    all_transcripts = {}

    for folder in os.listdir(dataset_dir):
        folder_path = os.path.join(dataset_dir, folder)
        if os.path.isdir(folder_path) and folder.endswith("_P"):
            for file in os.listdir(folder_path):
                if file.endswith("_TRANSCRIPT.csv"):
                    session_id = folder.split("_")[0]
                    file_path = os.path.join(folder_path, file)

                    try:
                        df = pd.read_csv(file_path)
                        df.columns = df.columns.str.strip().str.lower()

                        if 'speaker' in df.columns and 'value' in df.columns:
                            participant_lines = df[df['speaker'] == 'Participant']['value'].dropna().tolist()
                            if participant_lines:
                                all_transcripts[session_id] = participant_lines
                            else:
                                print(f" Không có lời thoại 'Participant' ở session {session_id}")
                        else:
                            print(f" Session {session_id} thiếu cột 'speaker' hoặc 'value'.")

                    except Exception as e:
                        print(f" Lỗi khi đọc {file_path}: {e}")
    
    return all_transcripts


In [6]:
import re

def clean_text(text):
    text = text.lower()
    text = re.sub(r'\[.*?\]', '', text)  # bỏ [noise], [laughter], ...
    text = re.sub(r'[^a-zA-Z0-9\s]', '', text)  # bỏ dấu câu
    text = re.sub(r'\s+', ' ', text).strip()
    return text


In [7]:
def generate_embeddings(all_transcripts):
    all_text_embeddings = {}

    for session_id, lines in all_transcripts.items():
        cleaned = [clean_text(t) for t in lines if isinstance(t, str)]
        full_text = " ".join(cleaned)

        try:
            inputs = tokenizer(full_text, return_tensors="pt", truncation=True, max_length=512)
            with torch.no_grad():
                outputs = model(**inputs)
                embedding = outputs.last_hidden_state.mean(dim=1).squeeze().numpy()
                all_text_embeddings[session_id] = embedding

        except Exception as e:
            print(f" Lỗi embedding session {session_id}: {e}")
            continue

    return all_text_embeddings


In [8]:
# 1. Duyệt và thu lời nói
all_transcripts = load_all_transcripts(dataset_dir)

# Kiểm tra 1 session đầu tiên
for k, v in all_transcripts.items():
    print(f" Session {k}: {len(v)} câu nói")
    print("   Ví dụ:", v[:3])
    break

# 2. Sinh embedding
all_text_embeddings = generate_embeddings(all_transcripts)

# 3. Lưu ra file
save_path = r"C:\Users\Acer\0. CAPSTONE\embeddings\text_embeddings\all_text_embeddings.npy"
np.save(save_path, all_text_embeddings)
print(f" Đã lưu toàn bộ embedding tại: {save_path}")
print(f" Số session đã xử lý: {len(all_text_embeddings)}")


 Session 301: 104 câu nói
   Ví dụ: ['thank you', 'mmm k', "i'm doing good thank you"]
 Đã lưu toàn bộ embedding tại: C:\Users\Acer\0. CAPSTONE\embeddings\text_embeddings\all_text_embeddings.npy
 Số session đã xử lý: 11


II. Trích embeddings từ audio

In [9]:
from pathlib import Path

In [10]:
# Đường dẫn thư mục
dataset_dir = Path("C:/Users/Acer/0. CAPSTONE/dataset")
output_path = dataset_dir.parent / "embeddings" / "audio_embeddings" / "all_audio_embeddings.npy"
output_path.parent.mkdir(parents=True, exist_ok=True)

# Cột cho từng loại đặc trưng (không có header nên cần tự đặt)
covarep_columns = [
    'F0', 'VUV', 'NAQ', 'QOQ', 'H1H2', 'PSP', 'MDQ', 'peakSlope', 'Rd', 'Rd_conf', 'creak',
    *['MCEP_' + str(i) for i in range(25)],
    *['HMPDM_' + str(i) for i in range(25)],
    *['HMPDD_' + str(i) for i in range(13)]
]

formant_columns = ['F1', 'F2', 'F3', 'F4', 'F5']


In [11]:
def extract_audio_embedding(session_dir: Path):
    session_id = session_dir.name.split("_")[0]
    covarep_path = session_dir / f"{session_id}_COVAREP.csv"
    formant_path = session_dir / f"{session_id}_FORMANT.csv"

    if not covarep_path.exists() or not formant_path.exists():
        return None
# Đọc không có header, gán tên thủ công
    cov = pd.read_csv(covarep_path, header=None, names=covarep_columns)
    frm = pd.read_csv(formant_path, header=None, names=formant_columns)

    # Lọc chỉ voiced: VUV == 1
    cov = cov[cov['VUV'] == 1]
    # Bỏ các dòng scrubbed toàn zero
    cov = cov.loc[~(cov == 0).all(axis=1)]
    frm = frm.loc[~(frm == 0).all(axis=1)]

    # Xóa cột không cần thiết
    cov = cov.drop(columns=['frameTime','VUV'], errors='ignore')
 # Gộp lại và tính thống kê
    combined = pd.concat([cov.reset_index(drop=True), frm.reset_index(drop=True)], axis=1)

    if combined.shape[0] == 0:
        return None

    stats = pd.concat([
        combined.mean(),
        combined.std(),
        combined.min(),
        combined.max()
    ])
    return stats.values.astype(np.float32)


In [12]:
audio_embeddings = {}
for session_dir in tqdm(dataset_dir.glob("*_P")):
    emb = extract_audio_embedding(session_dir)
    if emb is not None:
        session_id = session_dir.name.split("_")[0]
        audio_embeddings[session_id] = emb

np.save(output_path, audio_embeddings)
print(f" Đã lưu {len(audio_embeddings)} audio embeddings tại:\n{output_path}")


11it [00:14,  1.28s/it]

 Đã lưu 11 audio embeddings tại:
C:\Users\Acer\0. CAPSTONE\embeddings\audio_embeddings\all_audio_embeddings.npy


In [13]:
# Load lại file đã lưu
loaded_embeddings = np.load(output_path, allow_pickle=True).item()

# Lấy danh sách session_id, chỉ lấy 7 session đầu tiên
session_ids = sorted(loaded_embeddings.keys())[:7]
print(f" Tổng số session đã lưu: {len(loaded_embeddings)}\n")

# In thống kê của 7 session đầu
for session_id in session_ids:
    emb = loaded_embeddings[session_id]
    mean = np.mean(emb)
    std = np.std(emb)
    min_val = np.min(emb)
    max_val = np.max(emb)

    print(f"   Session: {session_id}")
    print(f"   Số chiều embedding: {emb.shape[0]}")
    print(f"   Mean: {mean:.4f}")
    print(f"   Std:  {std:.4f}")
    print(f"   Min:  {min_val:.4f}")
    print(f"   Max:  {max_val:.4f}\n")


 Tổng số session đã lưu: 11

   Session: 301
   Số chiều embedding: 312
   Mean: 122.8798
   Std:  631.1762
   Min:  -43.6700
   Max:  4945.0000

   Session: 302
   Số chiều embedding: 312
   Mean: 121.6400
   Std:  623.7429
   Min:  -39.9710
   Max:  4945.0000

   Session: 303
   Số chiều embedding: 312
   Mean: 122.5511
   Std:  629.5355
   Min:  -32.8720
   Max:  4945.0000

   Session: 304
   Số chiều embedding: 312
   Mean: 121.8909
   Std:  623.8619
   Min:  -40.0600
   Max:  4945.0000

   Session: 305
   Số chiều embedding: 312
   Mean: 123.6155
   Std:  635.9988
   Min:  -56.1010
   Max:  4945.0000

   Session: 306
   Số chiều embedding: 312
   Mean: 120.2738
   Std:  625.7987
   Min:  -44.6830
   Max:  4945.0000

   Session: 308
   Số chiều embedding: 312
   Mean: 122.3848
   Std:  624.9648
   Min:  -32.6710
   Max:  4945.0000

